# Week 3 — trained GPT-2 checkpoint evaluation (WEEK3EVAL)

This notebook evaluates the checkpoints produced by
`latent-agent-cost-week3-gpt2-train` **Version 1** on ProsQA
(n=500, seed 0, label `WEEK3EVAL`):

| order | checkpoint | condition | latent steps |
|---:|---|---|---:|
| 1 | stage-0 | `gpt2_cot` | 0 |
| 2 | final (stage-3-trained) | `gpt2_cot` | 0 |
| 3 | final (stage-3-trained) | `gpt2_latent` | **3 — do not change** |

**Before running**

1. Attach the training notebook's Version 1 Output as an input to
   this notebook. The locator cell searches `/kaggle/input`
   recursively; do not guess or hard-code its mounted path.
2. Set **Accelerator = GPU** (T4 x2 or P100) and **Internet = ON**.
3. Use **Save Version → Save & Run All (Commit)**.

**Hard-stop rule:** read the stage-0 `gpt2_cot` result first. It
is the explicit-fine-tune sanity baseline. If it is not a sane
ProsQA baseline, report that and do not interpret the final
latent-vs-CoT difference as a trained-latent effect; the paper
proceeds without the trained arm.


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU visible. Set Accelerator = GPU in the notebook settings and re-run.")
print("device:", torch.cuda.get_device_name(0))

Clone the project (fresh every run) and install its dependencies. `requirements.txt` already includes `tokenecon==0.2.0`; the one local pin filtered out is `torch` — Kaggle's preinstalled CUDA build must be kept, never replaced by the local CPU pin.

In [ ]:
!rm -rf latent-agent-cost
!git clone https://github.com/karmendra8386/latent-agent-cost
%cd latent-agent-cost
!grep -v '^torch' requirements.txt > /tmp/req-kaggle.txt
!pip install -q -r /tmp/req-kaggle.txt
!python -c "import torch; print('after install - torch:', torch.__version__, '| cuda:', torch.cuda.is_available())"

## Locate the trained checkpoints

The training output is an attached Kaggle input, whose mount
point is not stable enough to hard-code. This cell recursively
searches `/kaggle/input` (and the working directory) for the
exact checkpoint directory names written by
`scripts/train_coconut_gpt2.py`. A directory counts as a
checkpoint only if it contains `config.json`, a
`coconut_training.json` provenance sidecar, and model weights.

To search one known parent directory instead, set
`WEEK3_CKPT_ROOT` below to that directory before running the
cell. If the checkpoints are absent or ambiguous, the cell
fails loudly with attachment instructions.

In [ ]:
from pathlib import Path

# Optional override: a directory CONTAINING the checkpoint
# directories (for example, the parent `checkpoints/` directory
# from the training output). Leave empty to search Kaggle input.
WEEK3_CKPT_ROOT = ""

WEEK3EVAL_N = 500
WEEK3EVAL_SEED = 0
WEEK3EVAL_LATENT_STEPS = 3

# Recorded constants from the completed training run. These are
# context for the summary table, not values recomputed here.
TRAINING_RUNTIME_RECORDED = "5h51m49s"
TRAINING_STAGE_LOSSES_RECORDED = {0: 0.089, 1: 0.073, 2: 0.094, 3: 0.128}

STAGE0_DIR_NAME = "coconut-gpt2-prosqa-stage0"
FINAL_DIR_NAME = "coconut-gpt2-prosqa"
STAGE3_DIR_NAME = "coconut-gpt2-prosqa-stage3"


def _is_week3_checkpoint(path):
    if not path.is_dir():
        return False
    has_weights = any(path.glob("*.safetensors")) or any(
        path.glob("pytorch_model*.bin"))
    return ((path / "config.json").is_file()
            and (path / "coconut_training.json").is_file()
            and has_weights)


def _checkpoint_candidates(search_root, dir_name):
    candidates = []
    if not search_root.exists():
        return candidates
    if search_root.name == dir_name and _is_week3_checkpoint(search_root):
        candidates.append(search_root)
    for path in search_root.rglob(dir_name):
        if _is_week3_checkpoint(path):
            candidates.append(path)
    # De-duplicate while preserving deterministic order.
    unique = {}
    for path in candidates:
        try:
            key = str(path.resolve())
        except OSError:
            key = str(path)
        unique[key] = path
    return [unique[key] for key in sorted(unique)]


def _find_checkpoint(dir_names, label):
    if WEEK3_CKPT_ROOT:
        roots = [Path(WEEK3_CKPT_ROOT).expanduser()]
    else:
        roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    for dir_name in dir_names:
        candidates = []
        for root in roots:
            candidates.extend(_checkpoint_candidates(root, dir_name))
        # A name found under several roots may be the same mount;
        # de-duplicate by resolved path before judging ambiguity.
        unique = {}
        for path in candidates:
            try:
                key = str(path.resolve())
            except OSError:
                key = str(path)
            unique[key] = path
        candidates = [unique[key] for key in sorted(unique)]
        if len(candidates) == 1:
            return candidates[0], dir_name
        if len(candidates) > 1:
            joined = "\n  ".join(str(path) for path in candidates)
            raise RuntimeError(
                f"Ambiguous {label} checkpoint: found {len(candidates)} "
                f"valid directories named {dir_name}:\n  {joined}\n"
                "Set WEEK3_CKPT_ROOT to the parent directory of the "
                "intended training output and rerun this cell.")
    searched = ", ".join(str(root) for root in roots)
    expected = " or ".join(repr(name) for name in dir_names)
    raise FileNotFoundError(
        f"Could not find a valid {label} checkpoint ({expected}) under "
        f"{searched}.\n\n"
        "The training checkpoints are NOT in this notebook yet. Attach "
        "the Version 1 Output of `latent-agent-cost-week3-gpt2-train` "
        "as a notebook input (Add Input -> Notebook Output / Your Work, "
        "then select that training run's Version 1 output), and rerun "
        "from this cell. Each checkpoint directory must contain "
        "config.json, coconut_training.json, and model weights. "
        "Alternatively, set WEEK3_CKPT_ROOT above to a directory that "
        "contains the checkpoint directories and rerun this cell.")


STAGE0_PATH, _stage0_name = _find_checkpoint(
    (STAGE0_DIR_NAME,), "stage-0")
FINAL_PATH, FINAL_DIR_USED = _find_checkpoint(
    (FINAL_DIR_NAME, STAGE3_DIR_NAME), "final / stage-3")

STAGE0_CKPT = str(STAGE0_PATH)
FINAL_CKPT = str(FINAL_PATH)
FINAL_CHECKPOINT_KIND = (
    "final checkpoint" if FINAL_DIR_USED == FINAL_DIR_NAME
    else "stage-3 checkpoint snapshot")

print("stage-0 checkpoint:", STAGE0_CKPT)
print(f"final checkpoint ({FINAL_CHECKPOINT_KIND}):", FINAL_CKPT)
print("recorded training runtime:", TRAINING_RUNTIME_RECORDED)
print("recorded stage losses:", TRAINING_STAGE_LOSSES_RECORDED)

## Pre-flight — mandatory, before any 500-attempt cell

This cell loads both checkpoints and runs **one short generation
from each evaluated path** (stage-0 explicit CoT and final-checkpoint
latent, with the mandatory 3 latent steps). It also checks every
ProsQA prompt in the eval slice against GPT-2's 1,024-position
limit. If this cell fails, **stop**: do not start the 500-attempt
cells below.

Why this exists: Version 1 of this notebook died with a CUDA
device-side assert because the harness's 768-token generation
request could carry a long ProsQA prompt past GPT-2's
1,024-position embedding table (the first test prompt alone is
591 tokens). The harness now clamps each attempt's generation
budget to the positions actually remaining; this cell proves the
checkpoints, tokenizer, context guard, and both generation paths
work before the full run spends real time.


In [ ]:
import sys

sys.path.insert(0, ".")

import torch

from latent_cost.data import load_prosqa
from latent_cost.harness import build_messages
from latent_cost.models import (HFRunner, bounded_max_new_tokens,
                                model_context_limit)

_preflight_tasks = load_prosqa(limit=WEEK3EVAL_N)
_preflight_messages = build_messages(_preflight_tasks[0])

# Stage-0 checkpoint: check all 500 prompts against the model's
# position limit, then run one short explicit generation.
_stage0_runner = HFRunner(STAGE0_CKPT, dtype="float32", device="cuda")
_stage0_limit = model_context_limit(_stage0_runner.model)
_stage0_prompt_lengths = [
    len(_stage0_runner.tokenizer(
        _stage0_runner._prompt_text(build_messages(_task)))["input_ids"])
    for _task in _preflight_tasks]
assert _stage0_limit is not None and max(_stage0_prompt_lengths) < _stage0_limit
_stage0_worst_budget = bounded_max_new_tokens(
    768, prompt_tokens=max(_stage0_prompt_lengths),
    context_limit=_stage0_limit, reserved_positions=3)
_g = _stage0_runner.generate(
    _preflight_messages, seed=0, max_new_tokens=8, mode="explicit")
print(f"PREFLIGHT stage-0 gpt2_cot: prompt_tokens={_g.prompt_tokens}, "
      f"context_limit={_stage0_limit}, "
      f"max_prompt_tokens={max(_stage0_prompt_lengths)}, "
      f"worst_case_budget_with_3_reserved={_stage0_worst_budget}, "
      f"completion_tokens={_g.completion_tokens}")
del _stage0_runner
torch.cuda.empty_cache()

# Final checkpoint: one short latent generation at the trained
# budget of exactly 3 latent steps.
_final_runner = HFRunner(FINAL_CKPT, dtype="float32", device="cuda")
_final_limit = model_context_limit(_final_runner.model)
_g = _final_runner.generate(
    _preflight_messages, seed=0, max_new_tokens=8, mode="latent",
    latent_steps=WEEK3EVAL_LATENT_STEPS)
assert _g.latent_steps == WEEK3EVAL_LATENT_STEPS
print(f"PREFLIGHT final gpt2_latent: prompt_tokens={_g.prompt_tokens}, "
      f"context_limit={_final_limit}, latent_steps={_g.latent_steps}, "
      f"completion_tokens={_g.completion_tokens}")
del _final_runner
torch.cuda.empty_cache()

print("PREFLIGHT PASSED — checkpoints load, context guard is active, "
      "and both generation paths run. The 500-attempt cells may start.")


## 1 — Stage-0 checkpoint, explicit CoT

This is the sanity baseline and must be read before either
final-checkpoint number is interpreted.

In [ ]:
!python scripts/run_baseline.py --dataset prosqa --n 500 --seeds 0 --model "{STAGE0_CKPT}" --condition gpt2_cot --device cuda --dtype float32 --out results/week3eval_stage0_gpt2_cot_prosqa --label WEEK3EVAL

### Stage-0 sanity gate — read this before continuing

The next cell verifies the stage-0 attempts log (500 attempts,
seed 0, every attempt's `latent_steps` is 0) and prints its
accuracy and $/solve. No numeric sanity threshold was
pre-registered in the project record, so this notebook does not
invent one: judge whether the explicit fine-tune is a credible
ProsQA baseline. If it is degenerate or otherwise not sane,
apply the hard-stop rule in the summary and in `RUNBOOK.md` —
report the failed sanity gate and do not over-interpret the
trained latent comparison.

In [ ]:
import csv
import json
from pathlib import Path

_stage0_prefix = Path("results/week3eval_stage0_gpt2_cot_prosqa")
with _stage0_prefix.with_suffix(".csv").open(newline="") as fh:
    _stage0_row = next(csv.DictReader(fh))
_stage0_records = [
    json.loads(line)
    for line in _stage0_prefix.with_suffix(".attempts.jsonl")
    .read_text().splitlines() if line.strip()]

assert len(_stage0_records) == WEEK3EVAL_N == int(_stage0_row["attempts"])
assert {r["seed"] for r in _stage0_records} == {WEEK3EVAL_SEED}
assert {r["condition"] for r in _stage0_records} == {"gpt2_cot"}
assert {r["latent_steps"] for r in _stage0_records} == {0}

print("STAGE-0 SANITY GATE — READ BEFORE INTERPRETING WEEK3EVAL")
print(f"accuracy: {float(_stage0_row['accuracy']):.3f}")
print(f"usd_per_solve: {float(_stage0_row['usd_per_solve'])}")
print(f"attempts-log check: {WEEK3EVAL_N} attempts, seed "
      f"{WEEK3EVAL_SEED}, latent_steps all 0")

## 2 — Final checkpoint, explicit CoT

Same trained weights as the latent run below; only the
generation mode differs.

In [ ]:
!python scripts/run_baseline.py --dataset prosqa --n 500 --seeds 0 --model "{FINAL_CKPT}" --condition gpt2_cot --device cuda --dtype float32 --out results/week3eval_final_gpt2_cot_prosqa --label WEEK3EVAL

## 3 — Final checkpoint, latent

**`--latent-steps 3` is mandatory.** The final curriculum stage
trained the checkpoint to consume 3 latent feedback positions;
any other value evaluates a different interface.

In [ ]:
!python scripts/run_baseline.py --dataset prosqa --n 500 --seeds 0 --model "{FINAL_CKPT}" --condition gpt2_latent --latent-steps 3 --device cuda --dtype float32 --out results/week3eval_final_gpt2_latent_prosqa --label WEEK3EVAL

## Side-by-side summary

Writes `results/week3eval_summary.csv` and
`results/week3eval_summary.md`. Before tabulating, this cell
verifies from each attempts log — not from the commands above —
that the run has 500 seed-0 attempts, the expected condition,
and the expected latent-step count (0 for both CoT runs, exactly
3 for the latent run).

In [ ]:
import csv
import json
import math
from pathlib import Path

_summary_runs = [
    ("Stage 0", "stage-0 checkpoint", STAGE0_CKPT, "gpt2_cot", 0,
     Path("results/week3eval_stage0_gpt2_cot_prosqa")),
    ("Final (stage-3-trained)", FINAL_CHECKPOINT_KIND, FINAL_CKPT,
     "gpt2_cot", 0,
     Path("results/week3eval_final_gpt2_cot_prosqa")),
    ("Final (stage-3-trained)", FINAL_CHECKPOINT_KIND, FINAL_CKPT,
     "gpt2_latent", WEEK3EVAL_LATENT_STEPS,
     Path("results/week3eval_final_gpt2_latent_prosqa")),
]

_summary_fields = [
    "checkpoint", "checkpoint_kind", "checkpoint_path", "condition",
    "attempts", "solves", "accuracy", "usd_per_solve",
    "latency_per_solve_median_s", "latency_per_solve_iqr_s",
    "flops_per_solve", "tokens_per_solve", "total_tokens_per_solve",
    "auditability",
]
_summary_rows = []

for (_label, _kind, _ckpt, _condition, _expected_latent,
     _prefix) in _summary_runs:
    with _prefix.with_suffix(".csv").open(newline="") as fh:
        _row = next(csv.DictReader(fh))
    _records = [
        json.loads(line)
        for line in _prefix.with_suffix(".attempts.jsonl")
        .read_text().splitlines() if line.strip()]
    assert len(_records) == WEEK3EVAL_N == int(_row["attempts"]), _prefix
    assert {r["seed"] for r in _records} == {WEEK3EVAL_SEED}, _prefix
    assert {r["condition"] for r in _records} == {_condition}, _prefix
    assert {r["latent_steps"] for r in _records} == {
        _expected_latent}, (
            f"{_prefix}: latent_steps in attempts log are not all "
            f"{_expected_latent}")
    _summary_rows.append({
        "checkpoint": _label,
        "checkpoint_kind": _kind,
        "checkpoint_path": _ckpt,
        "condition": _condition,
        "attempts": _row["attempts"],
        "solves": _row["solves"],
        "accuracy": _row["accuracy"],
        "usd_per_solve": _row["usd_per_solve"],
        "latency_per_solve_median_s":
            _row["latency_per_solve_median_s"],
        "latency_per_solve_iqr_s": _row["latency_per_solve_iqr_s"],
        "flops_per_solve": _row["flops_per_solve"],
        "tokens_per_solve": _row["tokens_per_solve"],
        "total_tokens_per_solve": _row["total_tokens_per_solve"],
        "auditability": _row["auditability"],
    })

_summary_csv = Path("results/week3eval_summary.csv")
_summary_md = Path("results/week3eval_summary.md")
with _summary_csv.open("w", newline="") as fh:
    _writer = csv.DictWriter(fh, fieldnames=_summary_fields)
    _writer.writeheader()
    _writer.writerows(_summary_rows)


def _fmt_summary_value(key, value):
    number = float(value)
    if key == "accuracy":
        return f"{number:.3f}"
    if key == "usd_per_solve":
        return "inf" if math.isinf(number) else f"{number:.6f}"
    if key == "flops_per_solve":
        return "inf" if math.isinf(number) else f"{number:.3e}"
    if key in {"latency_per_solve_median_s",
               "latency_per_solve_iqr_s", "auditability"}:
        return "inf" if math.isinf(number) else f"{number:.4f}"
    return str(value)


_stage0_accuracy = float(_summary_rows[0]["accuracy"])
_lines = [
    f"# WEEK3EVAL summary — trained GPT-2 checkpoints (ProsQA, "
    f"n={WEEK3EVAL_N}, seed {WEEK3EVAL_SEED})",
    "",
    "**Hard-stop gate:** read the stage-0 `gpt2_cot` row first "
    f"(accuracy {_stage0_accuracy:.3f}). If that explicit "
    "fine-tune is not a sane ProsQA baseline, report the failed "
    "sanity gate and do not interpret the final latent-vs-CoT "
    "difference as a trained-latent effect.",
    "",
    "**Training context (recorded constants from the completed "
    "Kaggle training run, not recomputed in this notebook):** "
    f"runtime {TRAINING_RUNTIME_RECORDED}; stage losses "
    + ", ".join(
        f"stage {stage} {loss:.3f}"
        for stage, loss in TRAINING_STAGE_LOSSES_RECORDED.items())
    + ".",
    "",
    "| " + " | ".join(_summary_fields) + " |",
    "|" + "---|" * len(_summary_fields),
]
for _row in _summary_rows:
    _lines.append("| " + " | ".join(
        _fmt_summary_value(key, _row[key])
        if key not in {
            "checkpoint", "checkpoint_kind", "checkpoint_path",
            "condition", "attempts", "solves"}
        else str(_row[key])
        for key in _summary_fields) + " |")

_summary_md.write_text("\n".join(_lines) + "\n")
print(_summary_md.read_text())
print(f"wrote {_summary_csv} and {_summary_md}")

Show the individual five-metric tables and the summary files for retrieval.

In [ ]:
!ls -la results/week3eval_*
!cat results/week3eval_stage0_gpt2_cot_prosqa.md
!cat results/week3eval_final_gpt2_cot_prosqa.md
!cat results/week3eval_final_gpt2_latent_prosqa.md
!cat results/week3eval_summary.md